# Warp-Ingest — the PDF pipeline, step by step

Warp-Ingest is a **deterministic, rule-based PDF parser**. This notebook walks the
pure-Python pipeline end to end on a sample PDF:

```
PDF --pdfplumber--> per-word boxes + fonts --> Tika-format XHTML --> visual_ingestor --> blocks / JSON / HTML / OpenContracts
```

No Java, no Tika server, no OCR binary is required (scanned pages route to the
optional `rapidocr` backend automatically). Run `uv sync --group dev` first.

In [ ]:
from warp_ingest.ingestor import pdf_ingestor

PDF_PATH = '../files/pdf/8k.pdf'   # any PDF on disk

## 1. Front-end: PDF -> Tika-format XHTML

`parse_pdf` runs the pdfplumber front-end and returns the intermediate XHTML the
layout engine consumes. Each `<p>` is one visual line carrying per-word
`word-start/end-positions` and `word-fonts` in absolute, top-left-origin PDF points.

In [ ]:
xhtml = pdf_ingestor.parse_pdf(PDF_PATH, {})
print(xhtml['content'][:1500])

## 2. Layout engine: XHTML -> typed blocks

`parse_blocks` feeds the XHTML through `visual_ingestor.Doc`, which groups the
visual lines into typed blocks (`header`, `para`, `list_item`, `table_row`),
detects tables, strips repeating headers/footers and fixes reading order.

In [ ]:
blocks, *_ = pdf_ingestor.parse_blocks(xhtml)
print(f'{len(blocks)} blocks')
for b in blocks[:12]:
    print(f"{b['block_type']:11} | {b['block_text'][:80]}")

## 3. Render formats

`PDFIngestor` produces the `json` / `html` / `all` renderings used by the service's
`/api/parse?render_format=...` endpoint.

In [ ]:
ingestor = pdf_ingestor.PDFIngestor(PDF_PATH, {'render_format': 'all'})
print('pages:', ingestor.return_dict['num_pages'] + 1)
print('sections:', sum(1 for b in ingestor.blocks if b['block_type'] == 'header'))

## 4. OpenContracts structural export

`parse_to_opencontracts` emits the layout-aware export OpenContracts ingests:
PAWLS word tokens, one structural annotation per block, and the
`parent_id` heading hierarchy as explicit relationships. See
[docs/opencontracts_export_format.md](../docs/opencontracts_export_format.md).

In [ ]:
export = pdf_ingestor.parse_to_opencontracts(PDF_PATH)
print('page_count :', export['page_count'])
labels = [a['annotationLabel'] for a in export['labelled_text']]
from collections import Counter
print('labels     :', dict(Counter(labels)))